# New artifact

This notebook lives on the lab's own disk and is gone when the lab scales down. Declaring with `commit=True` keeps it: the last saved version is copied to the declared artifact's folder as `declaring.ipynb`. (This cell will be replaced with the artifact path.)

In [ ]:
import lab
from artifacts.core.artifact import Resources
from artifacts.core.SGD.training import LoopConfig, LRSchedule, OptimizerParameters, TrainingParameters
from artifacts.mappeddataset import MappedDataSet
from artifacts.sources import SourceURL
from artifacts.stages.pretraining import Pretraining
from artifacts.tokenizers.bpe import Tokenizer as BPETokenizer
from models.transformer import Transformer
from profiling.profile_events import aggregate_kernels, display_steps, events_to_rows, launch_chain
from profiling.step_report import build_report
from profiling.util import run_profile

In [ ]:
HF = "https://huggingface.co/datasets/roneneldan/TinyStories/resolve/main"
train = SourceURL(name="tinystories-train", url=f"{HF}/TinyStoriesV2-GPT4-train.txt")
valid = SourceURL(name="tinystories-valid", url=f"{HF}/TinyStoriesV2-GPT4-valid.txt")

tokenizer = BPETokenizer(
    vocab_size=10000,
    special_tokens=("<|endoftext|>",),
    sources=(valid,),
    allocated_resources=Resources(cpu=4),
)

dataset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=(train,),
    valid_sources=(valid,),
)

pretraining = Pretraining(
    run_id="my-run",
    dataset=dataset,
    model=Transformer(
        vocab_size=tokenizer.vocab_size,
        d_model=64,
        d_ff=128,
        sequence_length=32,
        num_layers=4,
        num_heads=16,
        rope_theta=10000,
        dtype="torch.float32",
    ),
    training_parameters=TrainingParameters(
        total_steps=20000,
        batch_size=64,
        max_norm=1.0,
        lr_schedule=LRSchedule(
            max_learning_rate=3e-4,
            min_learning_rate=3e-5,
            warmup_iters=200,
            cosine_cycle_iters=5000,
        ),
        optimizer="torch.optim.AdamW",
        optimizer_parameters=OptimizerParameters(
            lr=3e-4,
            betas=(0.9, 0.95),
            weight_decay=0.1,
            eps=1e-8,
        ),
        seed=0,
    ),
    loop_config=LoopConfig(
        checkpoint_every=5000,
        val_every=200,
        gpu_check_every=200,
    ),
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
    tags={"experiment": ["my-experiment"]},
)

In [ ]:
pretraining.pprint()

## Declare

Preview first: it writes nothing. Save the notebook before committing, since the copy is the file on disk.

In [ ]:
report = lab.declare(pretraining, verbose=True)

In [ ]:
report = lab.declare(pretraining, verbose=True, commit=True)

## Benchmark

In [ ]:
captured = run_profile(
    pretraining.allocated_resources,
    model_declaration=pretraining.model,
    training_parameters=pretraining.training_parameters,
    batch_size=pretraining.training_parameters.batch_size,
    warmup_steps=5,
    profile_steps=4,
    autocast=None,  # PretrainJob trains without autocast
    kernel_events=True,
    module_scopes=True,
    peak_memory=True,
    trace_memory=True,
)

In [ ]:
report = build_report(captured, depth=3)
display_steps(report, steps=[3])

In [ ]:
step = max(captured["steps"])
rows = events_to_rows(captured["steps"][step]["events"])
groups, kernels, unattributed_kernels, flop_sources = aggregate_kernels(rows, depth=3)
display(groups[["kernel_count", "gpu_time_us", "gpu_span_us", "estimated_flops"]])

In [ ]:
display(kernels[["name", "phase", "module_fqn", "owner_fqn", "gpu_start_us", "duration_us"]])

In [ ]:
kernel_id = report["steps"][str(step)]["kernels"]["id"][0]
display(launch_chain(rows, kernel_id, format=True))